[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Реализуйте **causal (masked) self-attention** — механизм, благодаря которому GPT предсказывает следующий токен, не подсматривая в будущее.

Как и в обычном softmax attention, каждая позиция сравнивает запрос с ключами и смешивает значения. Отличие: позиция может смотреть **только на себя и более ранние позиции**.

## Зачем нужна маска

Во время обучения все позиции последовательности обрабатываются параллельно. Без маски позиция `i` увидела бы правильные будущие токены и задача next-token prediction стала бы нечестной. Верхний треугольник матрицы scores соответствует парам `j > i`; его заполняют `-inf` **до** softmax, поэтому запрещённые вероятности становятся нулевыми.

Для последовательности длины 4 допустимость выглядит так: строка 0 видит `[0]`, строка 1 — `[0,1]`, строка 2 — `[0,1,2]`, строка 3 — все позиции. Поэтому первая строка attention weights обязана быть `[1,0,0,0]`, а первый выход — равен `V[:, 0]`. Маска формы `(S,S)` может broadcast по batch.

### Поток форм

`Q, K: (B,S,D_k)` → scores `(B,S,S)` → masked scores `(B,S,S)` → weights `(B,S,S)` → `weights @ V` формы `(B,S,D_v)`. Масштаб остаётся $\sqrt{D_k}$.

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

## План реализации

1. Вычислите scaled dot-product scores.
2. Постройте булеву strict upper-triangular mask (`diagonal=1`) на том же устройстве.
3. Подмените запрещённые scores на отрицательную бесконечность.
4. Примените softmax по ключам и смешайте `V`.

## Быстрые проверки

- Для `S=1` результат равен `V`.
- Изменение `K` и `V` после позиции `t` не меняет outputs на позициях `<= t`.
- Первая строка результата не зависит от остальных токенов.
- Backward должен давать конечные градиенты.

## Частые ошибки

- Маскировать нижний, а не верхний треугольник.
- Маскировать после softmax: вероятности уже успели нормализоваться с будущими токенами.
- Использовать ноль вместо `-inf`: нулевой score всё ещё получает положительную вероятность.
- Создавать mask на CPU при входах на другом устройстве.

## Материалы для глубокого изучения

- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) — decoder masking в архитектуре Transformer.
- [PyTorch SDPA documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) — семантика `is_causal` и attention masks.
- [PyTorch Transformer building blocks](https://docs.pytorch.org/tutorials/intermediate/transformer_building_blocks.html) — формы, causal bias и современные реализации.

## Вопросы для самопроверки

1. Почему маска ставит `-inf`, а не 0?
2. Как доказать тестом, что модель не видит будущее?
3. Почему позиция может видеть саму себя?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    pass  # Replace this

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

In [ ]:
from torch_judge import check
check('causal_attention')